# TP5 - complete version (cifar100 part + final table)
Set the runtime to **GPU** (Runtime -> Change runtime type -> T4), then **Run all** once.
All four pretext tasks are trained on cifar100 with the **same number of epochs**, then compared with a linear classifier.
The cifar10 numbers come from the earlier runs (copied in the last cell).

In [1]:
EPOCHS = 3          # same budget for the 4 tasks, so the comparison is fair. raise it if you have time
SIMCLR_BATCH = 128  # 256 can run out of gpu memory on a T4 (resnet50 on 2 views)
# rough time on a T4: patch ~20 s/epoch, rotation ~2 min/epoch, context encoder ~2 min/epoch, simclr not measured

## 1. Setup

In [2]:
import torch
assert torch.cuda.is_available(), 'no gpu: Runtime -> Change runtime type -> T4 GPU, then run again'
print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
!if [ -d cs294-158-ssl ]; then rm -Rf cs294-158-ssl; fi
!git clone -q https://github.com/ckurtz/cs294-158-ssl

import os
os.chdir('cs294-158-ssl')

In [4]:
# drive keeps the checkpoints if the runtime dies
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/tp5_final'  # new folder on purpose, no old half-finished runs in it
os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


### cifar100
The official server is throttled (about 50 kB/s), so it comes from huggingface and gets converted back to the official torchvision files (same pixels and labels as the official archive, checked).

In [ ]:
%%writefile cifar100_hf.py
"""get cifar100 without the (very slow) official server.

the images come from the huggingface parquet files and are written back in the exact
layout torchvision expects (root/cifar-100-python/{train,test,meta}), same pixels and labels.
"""
import io
import json
import pickle
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image

HF_URL = 'https://huggingface.co/datasets/uoft-cs/cifar100/resolve/main/cifar100'
SPLITS = ('train', 'test')


def download_parquet(parquet_dir):
    parquet_dir = Path(parquet_dir)
    parquet_dir.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        target = parquet_dir / f'{split}.parquet'
        if not target.exists():
            urllib.request.urlretrieve(f'{HF_URL}/{split}-00000-of-00001.parquet', target)
    return parquet_dir


def build_cifar100(parquet_dir, root):
    """convert the parquet files to torchvision's pickle files, returns root/cifar-100-python"""
    parquet_dir, base = Path(parquet_dir), Path(root) / 'cifar-100-python'
    base.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        df = pd.read_parquet(parquet_dir / f'{split}.parquet')
        imgs = np.stack([np.array(Image.open(io.BytesIO(d['bytes'])).convert('RGB')) for d in df['img']])
        entry = {
            'data': imgs.transpose(0, 3, 1, 2).reshape(len(df), -1),  # N x 3072, R then G then B planes
            'fine_labels': df['fine_label'].tolist(),
            'coarse_labels': df['coarse_label'].tolist(),
        }
        with open(base / split, 'wb') as f:
            pickle.dump(entry, f)

    # the class names live in the parquet schema
    schema = pq.read_schema(parquet_dir / 'train.parquet')
    names = json.loads(schema.metadata[b'huggingface'])['info']['features']['fine_label']['names']
    with open(base / 'meta', 'wb') as f:
        pickle.dump({'fine_label_names': names}, f)
    return base


def prepare_cifar100(root='data/cifar100', cache_dir=None):
    """make root/cifar-100-python ready. cache_dir (e.g. on google drive) keeps the converted
    files so a fresh colab runtime doesn't have to download and convert again"""
    root = Path(root)
    base = root / 'cifar-100-python'
    if all((base / n).is_file() for n in (*SPLITS, 'meta')):
        return base

    cached = Path(cache_dir) / 'data_cifar100' if cache_dir else None
    if cached and cached.is_dir():
        shutil.copytree(cached, root, dirs_exist_ok=True)
    else:
        build_cifar100(download_parquet(root.parent / 'cifar100_hf'), root)
        if cached:
            shutil.copytree(root, cached, dirs_exist_ok=True)

    assert all((base / n).is_file() for n in (*SPLITS, 'meta')), 'cifar100 missing'
    return base


In [ ]:
from cifar100_hf import prepare_cifar100

prepare_cifar100('data/cifar100', cache_dir=DRIVE_DIR)
print('cifar100 ready')

## 2. Patch position task + cifar100 support (patches the library)

In [7]:
%%writefile patch_position.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class PatchPosition(nn.Module):
    """Pretext task from Doersch et al. 2015 (arXiv 1505.05192).

    Take a query patch (center of a 3x3 grid) and one of its 8 neighbours,
    the network has to guess which of the 8 positions the neighbour came from.
    """
    metrics = ['Loss', 'Acc1']
    metrics_fmt = [':.4e', ':6.2f']

    # cifar images (32x32) get upsampled to 64x64 on the gpu in encode/forward, then the grid is:
    # 3 patches of 16px + 2 gaps of 6px = 60px, so 2px of margin on each side for the jitter
    IMG_SIZE = 64
    PATCH = 16
    GAP = 6
    JITTER = 2

    # (row, col) offset of the neighbour in grid cells, the label is the index in this list
    NEIGHBOURS = [(-1, -1), (-1, 0), (-1, 1),
                  (0, -1),           (0, 1),
                  (1, -1),  (1, 0),  (1, 1)]

    def __init__(self, dataset, n_classes, color_drop=True):
        super().__init__()
        self.dataset = dataset
        self.n_classes = n_classes
        # color dropping is there to stop the net from cheating with chromatic aberration
        self.color_drop = color_drop

        # same conv trunk for the 2 patches (siamese) and for the full image at test time
        self.encoder = nn.Sequential(
            self._block(3, 64, 1), self._block(64, 64, 2),      # 16 -> 8
            self._block(64, 128, 1), self._block(128, 128, 2),  # 8 -> 4
            self._block(128, 256, 1), self._block(256, 256, 2), # 4 -> 2
        )
        # a 16x16 patch ends up as 2x2, a 64x64 image as 8x8 so pool it down to 2x2 as well
        self.pool = nn.AdaptiveAvgPool2d(2)
        self.latent_dim = 256 * 2 * 2

        self.head = nn.Sequential(
            nn.Linear(2 * self.latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, len(self.NEIGHBOURS)),
        )

    @staticmethod
    def _block(in_ch, out_ch, stride):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def construct_classifier(self):
        return nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(self.latent_dim, affine=False),
            nn.Linear(self.latent_dim, self.n_classes),
        )

    def _features(self, x):
        return torch.flatten(self.pool(self.encoder(x)), 1)

    def encode(self, images):
        # full images only, patches go through _features directly
        if images.shape[-1] != self.IMG_SIZE:
            images = F.interpolate(images, size=self.IMG_SIZE, mode='bilinear', align_corners=False)
        return self._features(images)

    def forward(self, images):
        batch_size = images.shape[0]
        if images.shape[-1] != self.IMG_SIZE:
            images = F.interpolate(images, size=self.IMG_SIZE, mode='bilinear', align_corners=False)
        query, neighbour, targets = self._make_pairs(images)

        # both patches go through the same encoder, so run them as one batch
        feats = self._features(torch.cat((query, neighbour), dim=0))
        f_query, f_neighbour = feats[:batch_size], feats[batch_size:]
        logits = self.head(torch.cat((f_query, f_neighbour), dim=1))

        loss = F.cross_entropy(logits, targets)
        acc = logits.argmax(dim=-1).eq(targets).float().mean() * 100.

        # the representation used for the linear classifier is the full image (no patches, no color drop)
        # no grad on it since the classifier is trained separately in the training loop
        with torch.no_grad():
            zs = self._features(images)

        return dict(Loss=loss, Acc1=acc), zs

    def _make_pairs(self, images):
        batch_size, device = images.shape[0], images.device
        stride = self.PATCH + self.GAP
        origin = (self.IMG_SIZE - (3 * self.PATCH + 2 * self.GAP)) // 2
        center = origin + stride  # top left corner of the middle cell

        targets = torch.randint(len(self.NEIGHBOURS), (batch_size,), device=device)
        offsets = torch.tensor(self.NEIGHBOURS, device=device)[targets]  # (B, 2)

        # independent jitter for each patch so the gap size can't be used as a clue
        jitter = torch.randint(-self.JITTER, self.JITTER + 1, (2, batch_size, 2), device=device)
        query_pos = center + jitter[0]
        neighbour_pos = center + offsets * stride + jitter[1]

        query = self._crop(images, query_pos)
        neighbour = self._crop(images, neighbour_pos)
        if self.color_drop:
            query, neighbour = self._drop_colors(query), self._drop_colors(neighbour)
        return query, neighbour, targets

    def _crop(self, images, pos):
        # pos is (B, 2) with the (row, col) of the top left corner of each patch
        batch_size, device = images.shape[0], images.device
        steps = torch.arange(self.PATCH, device=device)
        rows = (pos[:, 0:1] + steps)[:, None, :, None]
        cols = (pos[:, 1:2] + steps)[:, None, None, :]
        batch_idx = torch.arange(batch_size, device=device)[:, None, None, None]
        chan_idx = torch.arange(3, device=device)[None, :, None, None]
        return images[batch_idx, chan_idx, rows, cols]

    @staticmethod
    def _drop_colors(patches):
        # keep one random channel and replace the other two by gaussian noise (like in the paper)
        batch_size, device = patches.shape[0], patches.device
        keep = torch.randint(3, (batch_size,), device=device)
        mask = F.one_hot(keep, 3).to(patches.dtype)[:, :, None, None]  # (B, 3, 1, 1)

        kept = (patches * mask).sum(dim=1, keepdim=True)
        std = kept.std(dim=(1, 2, 3), keepdim=True)
        noise = torch.randn_like(patches) * std
        return patches * mask + noise * (1. - mask)

Writing patch_position.py


In [8]:
%%writefile apply_patches.py
"""Patch a fresh clone of cs294-158-ssl so it supports:
  - the new `patch_position` task (patch_position.py is copied into deepul_helper/tasks/)
  - CIFAR100 as a second dataset for all the tasks

usage: python apply_patches.py <path to cs294-158-ssl> <path to patch_position.py>
"""
import shutil
import sys
from pathlib import Path

MARKER = '# [tp5 patch]'

# the official cifar server is really slow, so cifar100 is rebuilt from the huggingface parquet files
# (same pixels and labels, but not the same bytes) so torchvision's md5 checks can't pass on it
LOCAL_CIFAR100 = """class LocalCIFAR100(datasets.CIFAR100):
    def _check_integrity(self):
        names = [name for name, _ in self.train_list + self.test_list]
        return all(osp.isfile(osp.join(self.root, self.base_folder, n)) for n in names)

    def _load_meta(self):
        import pickle
        with open(osp.join(self.root, self.base_folder, self.meta['filename']), 'rb') as f:
            data = pickle.load(f, encoding='latin1')
        self.classes = data[self.meta['key']]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
"""


def patch_file(path, edits):
    text = path.read_text()
    if MARKER in text:
        print(f'{path.name}: already patched, skipping')
        return
    for old, new, count in edits:
        assert text.count(old) == count, f'{path.name}: expected {count}x {old!r}, found {text.count(old)}'
        text = text.replace(old, new)
    path.write_text(f'{MARKER}\n' + text)
    print(f'{path.name}: patched')


def main(repo, patch_module):
    repo = Path(repo)
    pkg = repo / 'deepul_helper'

    shutil.copy(patch_module, pkg / 'tasks' / 'patch_position.py')
    print('patch_position.py copied')

    patch_file(pkg / 'tasks' / '__init__.py', [
        ('from .simclr import SimCLR', 'from .simclr import SimCLR\nfrom .patch_position import PatchPosition', 1),
    ])

    # cifar100 uses the same code path as cifar10 (same image size)
    patch_file(pkg / 'tasks' / 'rotation.py', [
        ("dataset == 'cifar10'", "dataset.startswith('cifar')", 3),
    ])
    patch_file(pkg / 'tasks' / 'simclr.py', [
        ("dataset == 'cifar10'", "dataset.startswith('cifar')", 1),
    ])

    patch_file(pkg / 'data.py', [
        ("dataset == 'cifar10'", "dataset.startswith('cifar')", 4),
        ("    elif task == 'rotation':",
         """    elif task == 'patch_position':
        # stays 32x32 here, the model upsamples to 64x64 on the gpu (the colab cpu only has 2 cores)
        norm = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
        if train:
            transform = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
        else:
            transform = transforms.Compose([transforms.ToTensor(), norm])
    elif task == 'rotation':""", 1),
        ("datasets.CIFAR10(osp.join('data', dataset), train=True,",
         "cifar_cls(osp.join('data', dataset), train=True,", 1),
        ("datasets.CIFAR10(osp.join('data', dataset), train=False,",
         "cifar_cls(osp.join('data', dataset), train=False,", 1),
        ("    elif dataset.startswith('cifar'):\n",
         "    elif dataset.startswith('cifar'):\n"
         "        cifar_cls = datasets.CIFAR10 if dataset == 'cifar10' else LocalCIFAR100\n", 1),
        ("def get_datasets(dataset, task):", LOCAL_CIFAR100 + "\n\ndef get_datasets(dataset, task):", 1),
    ])

    patch_file(pkg / 'demos.py', [
        ("    elif task == 'simclr':\n        model = SimCLR(dataset, n_classes, None)\n",
         "    elif task == 'simclr':\n        model = SimCLR(dataset, n_classes, None)\n"
         "    elif task == 'patch_position':\n        model = PatchPosition(dataset, n_classes)\n", 1),
    ])

    patch_file(repo / 'train_self_supervised_task.py', [
        ("from warmup_scheduler import GradualWarmupScheduler",
         "try:\n    from warmup_scheduler import GradualWarmupScheduler\nexcept ImportError:\n"
         "    GradualWarmupScheduler = None  # only needed with -u > 0", 1),
        ("num_workers=16", "num_workers=min(16, os.cpu_count() or 2)", 2),
        ("assert args.task in ['context_encoder', 'rotation', 'cpc', 'simclr']",
         "assert args.task in ['context_encoder', 'rotation', 'cpc', 'simclr', 'patch_position']", 1),
        ("    elif args.task == 'simclr':\n        model = SimCLR(args.dataset, n_classes, dist)\n",
         "    elif args.task == 'simclr':\n        model = SimCLR(args.dataset, n_classes, dist)\n"
         "    elif args.task == 'patch_position':\n        model = PatchPosition(args.dataset, n_classes)\n", 1),
    ])


if __name__ == '__main__':
    if len(sys.argv) != 3:
        sys.exit(__doc__)
    main(sys.argv[1], sys.argv[2])

Writing apply_patches.py


In [9]:
!python apply_patches.py . patch_position.py

# make the patched library the one that gets imported (cwd is cs294-158-ssl)
import sys
sys.path.insert(0, os.getcwd())

patch_position.py copied
__init__.py: patched
rotation.py: patched
simclr.py: patched
data.py: patched
demos.py: patched
train_self_supervised_task.py: patched


In [10]:
# bring back the checkpoints already saved on drive (if the runtime died before), trainings below skip those
for task in ['patch_position', 'rotation', 'context_encoder', 'simclr']:
    src = f'{DRIVE_DIR}/cifar100_{task}'
    if os.path.exists(f'{src}/model_best.pth.tar'):
        shutil.copytree(src, f'results/cifar100_{task}', dirs_exist_ok=True)
        print('restored from drive:', task)

## 3. Trainings on cifar100
The log only shows the epoch summary lines (`* LinearAcc@1 ... Loss ... Acc1 ...`), the per-iteration lines are filtered out.
Each training is copied to drive when it ends, and is skipped only if a real `model_best.pth.tar` is already there (an empty folder from an interrupted run does not count).

In [11]:
# patch position (yours), pretext Acc1 should go well above 12.5%
if not os.path.exists(f'{DRIVE_DIR}/cifar100_patch_position/model_best.pth.tar'):
    !python -u train_self_supervised_task.py -d cifar100 -t patch_position -e {EPOCHS} -b 128 2>&1 | grep --line-buffered -vE "^(Epoch|Test): *\\["
    !cp -r results/cifar100_patch_position {DRIVE_DIR}/

Starting process on GPU: 0
[rank0]:[W1028 22:06:51.942075007 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
 * LinearAcc@1 7.292 LinearAcc@5 22.456 Loss 2.033 Acc1 18.249
 * LinearAcc@1 6.861 LinearAcc@5 21.815 Loss 1.960 Acc1 23.157
 * LinearAcc@1 8.554 LinearAcc@5 25.341 

In [12]:
if not os.path.exists(f'{DRIVE_DIR}/cifar100_rotation/model_best.pth.tar'):
    !python -u train_self_supervised_task.py -d cifar100 -t rotation -e {EPOCHS} 2>&1 | grep --line-buffered -vE "^(Epoch|Test): *\\["
    !cp -r results/cifar100_rotation {DRIVE_DIR}/

Starting process on GPU: 0
[rank0]:[W1028 22:08:59.336923765 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
 * LinearAcc@1 15.174 LinearAcc@5 36.669 Loss 1.069 Acc1 53.841
 * LinearAcc@1 20.823 LinearAcc@5 44.511 Loss 0.988 Acc1 58.408
 * LinearAcc@1 28.666 LinearAcc@5 57.2

In [13]:
# no run script for this one in the repo, optimizer / lr are a guess, check that the loss goes down
if not os.path.exists(f'{DRIVE_DIR}/cifar100_context_encoder/model_best.pth.tar'):
    !python -u train_self_supervised_task.py -d cifar100 -t context_encoder -e {EPOCHS} -o adam --lr 2e-4 2>&1 | grep --line-buffered -vE "^(Epoch|Test): *\\["
    !cp -r results/cifar100_context_encoder {DRIVE_DIR}/

Starting process on GPU: 0
[rank0]:[W1028 22:16:13.286449042 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
 * LinearAcc@1 8.574 LinearAcc@5 25.701 Loss 0.275
 * LinearAcc@1 11.178 LinearAcc@5 30.298 Loss 0.240
 * LinearAcc@1 13.101 LinearAcc@5 33.894 Loss 0.225
[rank0]:[W1

In [14]:
# last one because it is the heaviest. batch <= 256 so no warmup (-u 0), lr 0.5 (half of the repo's 1.0 for batch 512) is a guess for a short run
if not os.path.exists(f'{DRIVE_DIR}/cifar100_simclr/model_best.pth.tar'):
    !python -u train_self_supervised_task.py -d cifar100 -t simclr -b {SIMCLR_BATCH} -e {EPOCHS} -o lars --lr 0.5 -w 1e-6 -u 0 2>&1 | grep --line-buffered -vE "^(Epoch|Test): *\\["
    !cp -r results/cifar100_simclr {DRIVE_DIR}/

Starting process on GPU: 0
[rank0]:[W1028 22:23:58.276594386 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
/content/cs294-158-ssl/deepul_helper/lars.py:92: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:1853.)
  buf.mul_(momentum).add_(actual_lr, d_p + weight_decay * p.data)
/usr/local/lib/python3.13/dist-packa

## 4. Final table
cifar100 is evaluated here (models that have no checkpoint are skipped). The cifar10 numbers are copied from the earlier run (linear classification on the saved checkpoints, patch position trained 100 epochs), not recomputed.

In [15]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from deepul_helper.demos import load_model_and_data, evaluate_classifier

all_tasks = ['context_encoder', 'rotation', 'simclr', 'patch_position']

def evaluate_all(tasks, dataset):
    rows = []
    for task in tasks:
        if not os.path.exists(f'results/{dataset}_{task}/model_best.pth.tar'):
            print(f'{task} on {dataset}: no checkpoint, skipping')
            continue
        # epochs really done by this checkpoint (so the report can't say the wrong number)
        epochs = torch.load(f'results/{dataset}_{task}/model_best.pth.tar', map_location='cpu', weights_only=False)['epoch']
        model, lin, train_loader, test_loader = load_model_and_data(task, dataset)
        tr1, tr5 = evaluate_classifier(model, lin, train_loader)
        te1, te5 = evaluate_classifier(model, lin, test_loader)
        rows.append(dict(task=task, dataset=dataset, epochs=epochs, train_top1=tr1, test_top1=te1, train_top5=tr5, test_top5=te5))
        # free the gpu before the next model
        del model, lin, train_loader, test_loader
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

res_cifar100 = evaluate_all(all_tasks, 'cifar100')
res_cifar100.round(2)  # the epochs column is what to write in the report

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


,task,dataset,epochs,train_top1,test_top1,train_top5,test_top5
0,context_encoder,cifar100,3,14.91,13.09,36.80,33.89
1,rotation,cifar100,3,27.45,28.69,55.93,57.27
2,simclr,cifar100,3,10.72,12.29,30.52,34.43
3,patch_position,cifar100,3,9.10,8.56,26.17,25.35


In [16]:
# cifar10 numbers from the earlier run (test set). patch position = 100 epochs there
res_cifar10 = pd.DataFrame([
    dict(task='context_encoder', dataset='cifar10', test_top1=45.77, test_top5=90.29),
    dict(task='rotation',        dataset='cifar10', test_top1=79.91, test_top5=99.12),
    dict(task='simclr',          dataset='cifar10', test_top1=92.84, test_top5=99.86),
    dict(task='patch_position',  dataset='cifar10', test_top1=59.92, test_top5=94.97),
])

results = pd.concat([res_cifar10, res_cifar100], ignore_index=True)
results.round(2).to_csv('results_all.csv', index=False)

table = results.pivot(index='task', columns='dataset', values='test_top1').reindex(all_tasks)
print(table.round(2))

# epochs really done on cifar100 (read from the checkpoints), to write in the report
print(res_cifar100[['task', 'epochs', 'test_top1', 'test_top5']].round(2).to_string(index=False))
missing = [x for x in all_tasks if x not in set(res_cifar100.get('task', []))]
print('missing on cifar100:', missing or 'none')

ax = table.plot(kind='bar', figsize=(8, 4), rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f')
ax.set_ylabel('linear probe test top-1 (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()

!cp results_all.csv comparison.png {DRIVE_DIR}/

dataset          cifar10  cifar100
task                              
context_encoder    45.77     13.09
rotation           79.91     28.69
simclr             92.84     12.29
patch_position     59.92      8.56
           task  epochs  test_top1  test_top5
context_encoder       3      13.09      33.89
       rotation       3      28.69      57.27
         simclr       3      12.29      34.43
 patch_position       3       8.56      25.35
missing on cifar100: none
